<a href="https://colab.research.google.com/github/cathrineq/python-ai-Tarasova-Kate/blob/main/notebooks/viz2_currency_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📊 Week 3: Visualization — Визуализация

## 📥 [0] Подготовка данных: клонирование и загрузка

**Что делаем:**
- Клонируем репозиторий `python-ai-Tarasova-Kate` в Colab
- Загружаем 2 CSV-файла из Wikidata:
  - `currency_rates.csv` — курсы валют (P2284) с метками времени (P585, P580)
  - `countries_currencies.csv` — страны + официальные валюты (P38, P1082, P2046)
- Очищаем данные:
  - 🔗 URL Wikidata → переименовываем (`URL`, `country_URL`, `currency_URL`) — **не удаляем!**
  - 🏷️ `*Label` → короткие имена (`currency`, `unit`, `country`)
  - 📊 Числовые поля → `float`, пропуски (`NaN`) **не заменяем на 0**
  - 📈 Измеряем заполненность `OPTIONAL`-полей перед очисткой

**Результат:**

📊 `df_rates` — курсы валют по времени
| Столбец | Описание |
|---------|----------|
| `URL` | Ссылка на валюту в Wikidata 🔑 |
| `currency` | Название валюты |
| `price` | Курс в евро (float, возможны NaN) |
| `year` / `startYear` | Год данных (float, возможны NaN) |
| `unit` / `unitSymbol` | Единица измерения и символ |

🌍 `df_countries` — страны и их валюты
| Столбец | Описание |
|---------|----------|
| `country_URL` | Ссылка на страну в Wikidata |
| `country` | Название страны |
| `currency_URL` | Ссылка на валюту 🔑 (ключ для `merge`) |
| `currency` | Название валюты |
| `population` / `area` | Население и площадь (float, возможны NaN) |

> 💡 **Важно:** Столбцы `URL` / `currency_URL` сохранены для:  
> 1) отладки аномалий (клик → запись в Wikidata)  
> 2) надёжного объединения таблиц через `pd.merge(on="currency_URL")`

In [2]:
# ============================================================
#  ЯЧЕЙКА 1: Подготовка данных (для viz1 и viz2)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Клонируем репозиторий (если ещё не клонирован)
repo = "python-ai-Tarasova-Kate"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/cathrineq/{repo}.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Загружаем сырые CSV-файлы
df_rates_raw = pd.read_csv("data/currency_rates.csv")
df_countries_raw = pd.read_csv("data/countries_currencies.csv")

# 3. Очистка и переименование столбцов для df_rates (как в задании 2)
df_rates = df_rates_raw.rename(columns={
    "currency": "URL",
    "currencyLabel": "currency",
    "unitLabel": "unit"
}, errors="ignore")

# Приводим числовые столбцы к float (NaN, если не число)
df_rates["price"] = pd.to_numeric(df_rates["price"], errors="coerce")
if "year" in df_rates.columns:
    df_rates["year"] = pd.to_numeric(df_rates["year"], errors="coerce")
if "startYear" in df_rates.columns:
    df_rates["startYear"] = pd.to_numeric(df_rates["startYear"], errors="coerce")

# 4. Очистка для df_countries (переименовываем URL и Label)
df_countries = df_countries_raw.rename(columns={
    "country": "country_URL",
    "currency": "currency_URL",
    "countryLabel": "country",
    "currencyLabel": "currency"
}, errors="ignore")

# Приводим числовые столбцы
df_countries["population"] = pd.to_numeric(df_countries["population"], errors="coerce")
df_countries["area"] = pd.to_numeric(df_countries["area"], errors="coerce")

# 5. Документируем выбросы по цене (очень большие значения)
outliers = df_rates[df_rates["price"] >= 1e8]  # 100 миллионов и больше
if len(outliers) > 0:
    print("⚠️ ВНИМАНИЕ: Обнаружены выбросы по цене (≥ 100 000 000 евро):")
    print(outliers[["URL", "currency", "price", "year", "unit"]].to_string())
    print("\nЭти выбросы могут искажать цветовую шкалу на графиках.")
    print("В viz1 мы их временно исключим (оставим только цены < 1e8).\n")
else:
    print("✅ Выбросов по цене не обнаружено.\n")

# 6. Создаём df_unique_countries (уникальные страны, агрегированные)
#    Это нужно для второй визуализации (карта с дугами), но можно создать сразу.
df_unique_countries = (
    df_countries
    .groupby("country_URL")
    .agg(
        country=("country", "first"),               # название страны
        n_currencies=("currency_URL", "nunique"),  # сколько разных валют было у страны
        population=("population", "max"),          # население (берём максимум, если несколько записей)
        area=("area", "max")                       # площадь
    )
    .reset_index()
)
print(f"✅ df_unique_countries: {len(df_unique_countries)} уникальных стран (из {len(df_countries)} строк)")

# 7. Краткая информация о данных (для проверки)
print("\n📊 df_rates:", df_rates.shape)
print("Столбцы:", df_rates.columns.tolist())
print("\n📊 df_countries:", df_countries.shape)
print("Столбцы:", df_countries.columns.tolist())
print("\n✅ Подготовка данных завершена.")

/content/python-ai-Tarasova-Kate
⚠️ ВНИМАНИЕ: Обнаружены выбросы по цене (≥ 100 000 000 евро):
                                           URL            currency         price    year                   unit
523   http://www.wikidata.org/entity/Q56349362  Суверенный боливар  1.000000e+08  2018.0  венесуэльский боливар
526   http://www.wikidata.org/entity/Q56349362  Суверенный боливар  1.000000e+08  2018.0  венесуэльский боливар
529   http://www.wikidata.org/entity/Q56349362  Суверенный боливар  1.000000e+08  2018.0  венесуэльский боливар
1417    http://www.wikidata.org/entity/Q260447       рентная марка  1.000000e+09  1923.0             рейхсмарка

Эти выбросы могут искажать цветовую шкалу на графиках.
В viz1 мы их временно исключим (оставим только цены < 1e8).

✅ df_unique_countries: 209 уникальных стран (из 803 строк)

📊 df_rates: (2346, 8)
Столбцы: ['URL', 'currency', 'shortName', 'price', 'year', 'startYear', 'unit', 'unitSymbol']

📊 df_countries: (803, 7)
Столбцы: ['country_URL', '

In [5]:
# ============================================================
#  ЯЧЕЙКА 2: Карта с кружками и дугами (исправленная версия 2)
# ============================================================

!pip install plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# ------------------- НАСТРОЙКИ (развилки) -------------------
CURRENCY_FILTER = "no_eur_usd"   # "all", "no_eur_usd", "historical"
ARC_LABEL = "shortName"          # "shortName", "currency", "none"
ARC_WIDTH_BY_SIZE = True

# ------------------- КООРДИНАТЫ СТРАН (словарь) -------------------
country_coords = {
    "Russia": (61.5240, 105.3188), "United States": (37.0902, -95.7129), "Germany": (51.1657, 10.4515),
    "France": (46.2276, 2.2137), "United Kingdom": (55.3781, -3.4360), "China": (35.8617, 104.1954),
    "Japan": (36.2048, 138.2529), "India": (20.5937, 78.9629), "Brazil": (-14.2350, -51.9253),
    "Canada": (56.1304, -106.3468), "Australia": (-25.2744, 133.7751), "Italy": (41.8719, 12.5674),
    "Spain": (40.4637, -3.7492), "Mexico": (23.6345, -102.5528), "Nigeria": (9.0820, 8.6753),
    "South Africa": (-30.5595, 22.9375), "Turkey": (38.9637, 35.2433), "Poland": (51.9194, 19.1451),
    "Ukraine": (48.3794, 31.1656), "Kazakhstan": (48.0196, 66.9237), "Belarus": (53.7098, 27.9534),
    "Finland": (61.9241, 25.7482), "Sweden": (60.1282, 18.6435), "Norway": (60.4720, 8.4689),
    "Denmark": (56.2639, 9.5018), "Netherlands": (52.1326, 5.2913), "Belgium": (50.8503, 4.3517),
    "Switzerland": (46.8182, 8.2275), "Austria": (47.5162, 14.5501), "Czech Republic": (49.8175, 15.4730),
    "Slovakia": (48.6690, 19.6990), "Hungary": (47.1625, 19.5033), "Romania": (45.9432, 24.9668),
    "Bulgaria": (42.7339, 25.4858), "Greece": (39.0742, 21.8243), "Portugal": (39.3999, -8.2245),
    "Ireland": (53.4129, -8.2439), "Latvia": (56.8796, 24.6032), "Lithuania": (55.1694, 23.8813),
    "Estonia": (58.5953, 25.0136), "Slovenia": (46.1512, 14.9955), "Croatia": (45.1000, 15.2000),
    "Serbia": (44.0165, 21.0059), "Albania": (41.1533, 20.1683), "Moldova": (47.4116, 28.3699),
    "Georgia": (42.3154, 43.3569), "Armenia": (40.0691, 45.0382), "Azerbaijan": (40.1431, 47.5769),
    "Uzbekistan": (41.3775, 64.5853), "Kyrgyzstan": (41.2044, 74.7661), "Tajikistan": (38.8610, 71.2761),
    "Turkmenistan": (38.9697, 59.5563), "Mongolia": (46.8625, 103.8467), "Egypt": (26.8206, 30.8025),
    "Kenya": (-1.2864, 36.8172), "Ethiopia": (9.1450, 40.4897), "Ghana": (7.9465, -1.0232),
    "Ivory Coast": (7.5400, -5.5471), "Senegal": (14.4974, -14.4524), "Cameroon": (3.8480, 11.5021),
    "Zimbabwe": (-19.0154, 29.1549), "Vietnam": (14.0583, 108.2772), "Thailand": (15.8700, 100.9925),
    "Indonesia": (-0.7893, 113.9213), "Malaysia": (4.2105, 101.9758), "Philippines": (12.8797, 121.7740),
    "Pakistan": (30.3753, 69.3451), "Bangladesh": (23.6850, 90.3563), "Nepal": (28.3949, 84.1240),
    "Sri Lanka": (7.8731, 80.7718), "Argentina": (-38.4161, -63.6167), "Chile": (-35.6751, -71.5430),
    "Colombia": (4.5709, -74.2973), "Peru": (-9.1900, -75.0152), "Venezuela": (6.4238, -66.5897),
    "Ecuador": (-1.8312, -78.1834), "Paraguay": (-23.4425, -58.4438), "Uruguay": (-32.5228, -55.7658),
    "Bolivia": (-16.2902, -63.5887), "South Korea": (35.9078, 127.7669), "North Korea": (40.3399, 127.5101),
    "Saudi Arabia": (23.8859, 45.0792), "United Arab Emirates": (23.4241, 53.8478), "Israel": (31.0461, 34.8516),
    "Iraq": (33.2232, 43.6793), "Iran": (32.4279, 53.6880), "Afghanistan": (33.9391, 67.7100),
    "Yemen": (15.5527, 48.5164), "Oman": (21.5126, 55.9233), "Kuwait": (29.3117, 47.4818),
    "Qatar": (25.3548, 51.1839), "Lebanon": (33.8547, 35.8623), "Syria": (34.8021, 38.9968),
    "Jordan": (30.5852, 36.2384), "Cyprus": (35.1264, 33.4299), "Malta": (35.9375, 14.3754),
    "Iceland": (64.9631, -19.0208), "Luxembourg": (49.8153, 6.1296), "Monaco": (43.7384, 7.4246),
    "Andorra": (42.5462, 1.6016), "Montenegro": (42.7087, 19.3744), "Bosnia and Herzegovina": (43.9159, 17.6791),
    "North Macedonia": (41.6086, 21.7453), "Burkina Faso": (12.2383, -1.5616), "Mali": (17.5707, -3.9962),
    "Niger": (17.6078, 8.0817), "Chad": (15.4542, 18.7322), "Sudan": (12.8628, 30.2176),
    "South Sudan": (6.8770, 31.3070), "Eritrea": (15.1794, 39.7823), "Djibouti": (11.8251, 42.5903),
    "Somalia": (5.1521, 46.1996), "Uganda": (1.3733, 32.2903), "Rwanda": (-1.9403, 29.8739),
    "Burundi": (-3.3731, 29.9189), "Tanzania": (-6.3690, 34.8888), "Mozambique": (-18.6657, 35.5296),
    "Malawi": (-13.2543, 34.3015), "Zambia": (-13.1339, 27.8493), "Angola": (-11.2027, 17.8739),
    "Gabon": (-0.8037, 11.6094), "Madagascar": (-18.7669, 46.8691), "Botswana": (-22.3285, 24.6849),
    "Namibia": (-22.9576, 18.4904), "Lesotho": (-29.6100, 28.2336)
}

# Словарь перевода русских названий стран в английские
name_translation = {
    "Россия": "Russia", "США": "United States", "Германия": "Germany", "Франция": "France",
    "Великобритания": "United Kingdom", "Китай": "China", "Япония": "Japan", "Индия": "India",
    "Бразилия": "Brazil", "Канада": "Canada", "Австралия": "Australia", "Италия": "Italy",
    "Испания": "Spain", "Мексика": "Mexico", "Нигерия": "Nigeria", "ЮАР": "South Africa",
    "Турция": "Turkey", "Польша": "Poland", "Украина": "Ukraine", "Казахстан": "Kazakhstan",
    "Беларусь": "Belarus", "Финляндия": "Finland", "Швеция": "Sweden", "Норвегия": "Norway",
    "Дания": "Denmark", "Нидерланды": "Netherlands", "Бельгия": "Belgium", "Швейцария": "Switzerland",
    "Австрия": "Austria", "Чехия": "Czech Republic", "Словакия": "Slovakia", "Венгрия": "Hungary",
    "Румыния": "Romania", "Болгария": "Bulgaria", "Греция": "Greece", "Португалия": "Portugal",
    "Ирландия": "Ireland", "Латвия": "Latvia", "Литва": "Lithuania", "Эстония": "Estonia",
    "Словения": "Slovenia", "Хорватия": "Croatia", "Сербия": "Serbia", "Албания": "Albania",
    "Молдова": "Moldova", "Грузия": "Georgia", "Армения": "Armenia", "Азербайджан": "Azerbaijan",
    "Узбекистан": "Uzbekistan", "Киргизия": "Kyrgyzstan", "Таджикистан": "Tajikistan",
    "Туркменистан": "Turkmenistan", "Монголия": "Mongolia", "Египет": "Egypt", "Кения": "Kenya",
    "Эфиопия": "Ethiopia", "Гана": "Ghana", "Кот-д'Ивуар": "Ivory Coast", "Сенегал": "Senegal",
    "Камерун": "Cameroon", "Зимбабве": "Zimbabwe", "Вьетнам": "Vietnam", "Таиланд": "Thailand",
    "Индонезия": "Indonesia", "Малайзия": "Malaysia", "Филиппины": "Philippines", "Пакистан": "Pakistan",
    "Бангладеш": "Bangladesh", "Непал": "Nepal", "Шри-Ланка": "Sri Lanka", "Аргентина": "Argentina",
    "Чили": "Chile", "Колумбия": "Colombia", "Перу": "Peru", "Венесуэла": "Venezuela",
    "Эквадор": "Ecuador", "Парагвай": "Paraguay", "Уругвай": "Uruguay", "Боливия": "Bolivia"
}

def get_coords(country_ru):
    """Возвращает (lat, lon) для русского названия страны"""
    en = name_translation.get(country_ru)
    if en is None:
        return None, None
    coords = country_coords.get(en)
    if coords is None:
        return None, None
    return coords[0], coords[1]

# Добавляем координаты в df_unique_countries
df_unique_countries["lat"] = None
df_unique_countries["lon"] = None
for idx, row in df_unique_countries.iterrows():
    lat, lon = get_coords(row["country"])
    df_unique_countries.at[idx, "lat"] = lat
    df_unique_countries.at[idx, "lon"] = lon

# Удаляем страны без координат
df_map = df_unique_countries.dropna(subset=["lat", "lon"]).copy()
print(f"Стран с координатами: {len(df_map)} из {len(df_unique_countries)}")

# ------------------- ПУЗЫРЬКОВАЯ КАРТА -------------------
fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lon=df_map["lon"],
    lat=df_map["lat"],
    text=df_map["country"],
    mode='markers',
    marker=dict(
        size=df_map["n_currencies"] * 1.5,
        color=df_map["n_currencies"],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Число валют<br>за всю историю"),
        line=dict(width=0.5, color='black')
    ),
    hoverinfo='text',
    hovertemplate='<b>%{text}</b><br>Валют: %{marker.color}<extra></extra>'
))

# ------------------- ДУГИ (валютные союзы) -------------------
# Группируем валюты по странам
currency_groups = df_countries.groupby("currency_URL")["country_URL"].apply(list).reset_index()
currency_groups["n_countries"] = currency_groups["country_URL"].apply(len)
currency_groups = currency_groups[(currency_groups["n_countries"] >= 2) & (currency_groups["n_countries"] <= 8)]

# Добавляем информацию о валюте (только те столбцы, которые есть в df_countries)
currency_info = df_countries.groupby("currency_URL").first().reset_index()[["currency_URL", "currency", "shortName"]]
currency_groups = currency_groups.merge(currency_info, on="currency_URL", how="left")

print("Доступные столбцы в currency_groups:", currency_groups.columns.tolist())

# Применяем фильтр
if CURRENCY_FILTER == "no_eur_usd":
    exclude = ["евро", "доллар США"]
    currency_groups = currency_groups[~currency_groups["currency"].str.lower().isin(exclude)]
elif CURRENCY_FILTER == "historical":
    if "year" in df_rates.columns:
        max_years = df_rates.groupby("currency")["year"].max().reset_index()
        max_years.columns = ["currency", "max_year"]
        currency_groups = currency_groups.merge(max_years, on="currency", how="left")
        currency_groups = currency_groups[currency_groups["max_year"] < 2000]

print(f"Валют для отрисовки дуг: {len(currency_groups)}")

# Рисуем дуги
for _, row in currency_groups.iterrows():
    countries = row["country_URL"]
    points = []
    for c_url in countries:
        c_row = df_map[df_map["country_URL"] == c_url]
        if len(c_row) == 0:
            continue
        lat = c_row.iloc[0]["lat"]
        lon = c_row.iloc[0]["lon"]
        points.append((lat, lon, c_row.iloc[0]["country"]))
    if len(points) < 2:
        continue
    for i in range(len(points)):
        for j in range(i+1, len(points)):
            lat1, lon1, name1 = points[i]
            lat2, lon2, name2 = points[j]
            width = max(1, min(5, row["n_countries"])) if ARC_WIDTH_BY_SIZE else 1.5
            # Подпись для дуги
            if ARC_LABEL == "shortName" and pd.notna(row.get("shortName")):
                label = str(row["shortName"])
            else:
                label = row["currency"][:10] if pd.notna(row.get("currency")) else "валюта"
            fig.add_trace(go.Scattergeo(
                lon=[lon1, lon2],
                lat=[lat1, lat2],
                mode='lines',
                line=dict(width=width, color='orange'),
                opacity=0.6,
                hoverinfo='text',
                hovertemplate=f'{label}<br>{name1} ↔ {name2}<extra></extra>',
                showlegend=False
            ))

# Настройка внешнего вида
fig.update_layout(
    title=dict(text="Количество исторических валют по странам и валютные союзы (оранжевые дуги)", x=0.5, font=dict(size=16)),
    geo=dict(
        showland=True, landcolor='lightgray',
        showocean=True, oceancolor='lightblue',
        showcountries=True, countrycolor='black',
        projection_type='natural earth'
    ),
    width=1200, height=700
)

fig.show()
print("✅ Карта построена.")
print(f"Всего стран с координатами: {len(df_map)}")
print(f"Валютных союзов (2-8 стран): {len(currency_groups)}")

Стран с координатами: 77 из 209
Доступные столбцы в currency_groups: ['currency_URL', 'country_URL', 'n_countries', 'currency', 'shortName']
Валют для отрисовки дуг: 21


✅ Карта построена.
Всего стран с координатами: 77
Валютных союзов (2-8 стран): 21
